# W boson mass

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import scipy
from pathlib import Path
from io import StringIO
import matplotlib.pyplot as plt
from scipy import optimize
from scipy.optimize import curve_fit
from scipy.stats import norm
from iminuit import cost as iminuit_cost, Minuit
from statsmodels.stats.moment_helpers import corr2cov

# Set a random seed for reproducibility
RANDOM_SEED = 42

### Extract data

In [15]:
text = Path("measurements.txt").read_text()

In [26]:
# Empty lists to hold data
data_lines = []
matrix_lines = []

# Go through all the lines in the text file
for line in text.splitlines():

    # Obtain the values for the measurements and uncertainties
    if "+/-" in line:
        value, rest = line.split("+/-", 1)
        uncertainty = rest.split()[0] # Spots before "MeV" or extra text
        data_lines.append(f"{value.strip()} {uncertainty}")

    # Obtain values for the correlation matrix
    fields = line.split()
    if len(fields) == 5:
        try:
            [float(field) for field in fields]
        except ValueError:
            continue
        matrix_lines.append(line)

data = np.loadtxt(StringIO("\n".join(data_lines)))
measurements, uncertainties = data.T
correlation = np.loadtxt(StringIO("\n".join(matrix_lines)))

In [27]:
print(measurements)
print(uncertainties)
print(correlation)

[80354.  80360.2 80366.6 80375.  80376.  80433.5]
[32.   9.9 15.9 23.  33.   6.4]
[[1.  0.5 0.5 0.5 0.5]
 [0.5 1.  0.5 0.5 0.5]
 [0.5 0.5 1.  0.5 0.5]
 [0.5 0.5 0.5 1.  0.5]
 [0.5 0.5 0.5 0.5 1. ]]


Can you find the best linear unbiased estimate of the combination? Can you show that this gives the same result as fitting a constant? What happens if you remove the outlier? How consistent are the outlier and the combination of the consistent results? We can express this in 'sigma' using "(outlier - combination)/sqrt(unc_outlier**2 + unc_combination**2)

What happens if you assume the consistent measurements all have a correlation of 0.5? How much larger is the uncertainty on the combination?
How much does the combined central value shift?
What is the consistency of the outlier and combination now (assume the outlier is fully uncorrelated with the other results)?

### Perform combinations
Perform the combinations of these datasets into a single measurements. Use the Best Linear Unbiased Estimator (BLUE)

In [84]:
# Number of measurements
no_measurements = len(measurements)

# Sum of 1 / sigma^2 of all the uncertainties 
inv_sum_weights = np.sum(1/uncertainties**2)

# Empty arrays to hold data
weights = np.zeros(no_measurements)
x_combinations = np.zeros(no_measurements)

# Calculate individual weights and normalise 
weights = (1 / uncertainties**2) / inv_sum_weights

# Calculate elements for x_comb
x_combinations = weights * measurements

# Combined uncertainty 1 / sigma_combined^2 = sum of 1 / sigma_i^2
combined_uncertainty = combined_uncertainty = np.sqrt(1 / inv_sum_weights)

# X Combinations
x_comb = float(np.sum(x_combinations))

In [85]:
print(f"weights:", weights)

weights: [0.02305506 0.24087727 0.09338389 0.04462832 0.02167895 0.5763765 ]


In [86]:
print(f"W boson mass combination:", x_comb)
print(f"Combined uncertainty:", combined_uncertainty)

W boson mass combination: 80403.9061396997
Combined uncertainty: 4.858845695581695


### Show the BLUE method gives the same result as fitting a constant to the data

In [108]:
# Define function
# Returns a full array of the same shape and type as given array 
def constant(x, c):
    return np.full_like(x, c, dtype = float)

In [109]:
x = np.arange(len(measurements)) 

popt, pcov = curve_fit(constant, x, measurements, sigma = uncertainties, absolute_sigma = True)

best_fit = popt[0]
fit_uncertainty = np.sqrt(pcov[0, 0])

In [111]:
print(f"W boson mass combination:", best_fit)
print(f"Combined uncertainty:", fit_uncertainty)

W boson mass combination: 80403.90613969973
Combined uncertainty: 4.858845681769849


#### Do this again but without the outlier

In [216]:
x_cut = np.arange(len(measurements[:5])) 
measurements_cut = measurements[:5]
uncertainties_cut = uncertainties[:5]

popt, pcov = curve_fit(constant, x_cut, measurements_cut, sigma = uncertainties[:5], absolute_sigma = True)

best_fit_cut = popt[0]
fit_uncertainty_cut = np.sqrt(pcov[0, 0])

In [217]:
print(f"W boson mass combination:", best_fit_cut)
print(f"Combined uncertainty:", fit_uncertainty_cut)

W boson mass combination: 80363.64112675173
Combined uncertainty: 7.465228119966647


### Calculate sigma of measurements from the combined measurements

In [226]:
# Obtain the outlier by inspection
outlier = measurements[5]
outlier_uncertainty = uncertainties[5]

# Calculate sigma compared to best fit data without outlier
sigma = float((outlier - best_fit_cut) / np.sqrt(outlier_uncertainty**2 + fit_uncertainty_cut**2))

    # recalculate combined uncertainty without uncertainty (done!)

In [227]:
print(f"Sigma of outlier", sigma)

Sigma of outlier 7.104469031284157


### Treat the uncertainties in each experiment as correlated

In [237]:
# Print correlation matrix
print(f"Correlation matrix:", correlation)

Correlation matrix: [[1.  0.5 0.5 0.5 0.5]
 [0.5 1.  0.5 0.5 0.5]
 [0.5 0.5 1.  0.5 0.5]
 [0.5 0.5 0.5 1.  0.5]
 [0.5 0.5 0.5 0.5 1. ]]


In [238]:
# Get first five measurements
x = measurements[:5]

In [239]:
# Calculate covariance matrix
covariance = corr2cov(correlation, uncertainties[:5])
print(covariance)

[[1024.     158.4    254.4    368.     528.   ]
 [ 158.4     98.01    78.705  113.85   163.35 ]
 [ 254.4     78.705  252.81   182.85   262.35 ]
 [ 368.     113.85   182.85   529.     379.5  ]
 [ 528.     163.35   262.35   379.5   1089.   ]]


In [240]:
# Define function to calculate the chi-squared
def chi_squared(c):
    D = x - c
    #return D @ np.linalg.solve(covariance, D)
    return D.T @ covariance**-2 @ D

In [241]:

x_estimate = np.mean(x)
res = scipy.optimize.minimize(chi_squared, x_estimate)

In [242]:
print(res)

  message: Optimization terminated successfully.
  success: True
   status: 0
      fun: -0.00686416468924998
        x: [ 8.036e+04]
      nit: 3
      jac: [-5.239e-10]
 hess_inv: [[ 5.479e+02]]
     nfev: 14
     njev: 7


In [253]:
uncertainty = np.sqrt(np.diag(res.hess_inv))

In [254]:
print(f"W boson mass", res.x[0])
print(f"Uncertainty:", uncertainty[0])

W boson mass 80364.5291191532
Uncertainty: 23.407458897698998


In [255]:
# compare c uncertainty here to other methods